# QLoRA Fine-tuning — self-healing-mlops-agent (Phase 1)

Groq(`qwen/qwen3.6-27b`) 대비 소형 FT 모델(Qwen2.5-7B-Instruct, 4bit QLoRA) 비교 실험.

**사전 준비**: Runtime → Change runtime type → T4 GPU 선택 후 저장, 그다음 이 노트북을 위에서부터 순서대로 실행.

준비물: `data/qlora/train.jsonl`(508건), `data/qlora/eval.jsonl`(614건) — 로컬 저장소에서 `python scripts/export_qlora_dataset.py`로 생성된 파일. 아래 셀에서 이 둘 파일을 업로드한다.

In [ ]:
# 1) 패키지 설치 (약 2~3분 소요)
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes


## 2) 학습 데이터 업로드
아래 셀을 실행하면 파일 선택 창이 뜼다 — `train.jsonl`과 `eval.jsonl` 둘 다 선택해서 업로드할 것.

In [ ]:
from google.colab import files
uploaded = files.upload()  # train.jsonl, eval.jsonl 선택
assert "train.jsonl" in uploaded and "eval.jsonl" in uploaded, "두 파일 다 업로드했는지 확인"


## 3) (권장) Google Drive 마운트
학습된 LoRA 어댑터를 Drive에 백업해두면 Colab 세션이 끊겨도(무료 티어는 12시간 제한/비활성 시 연결 끊김 가능) 결과가 날아가지 않는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/self-healing-mlops-agent/qlora_adapter"


## 4) 베이스 모델 로딩 (4bit, Unsloth)

T4(16GB)에 맞는 Qwen2.5-7B-Instruct 4bit 사전양자화 버전. **세션이 계속 끊기거나 느리면** 아래 `model_name`을 `"unsloth/Qwen2.5-3B-Instruct-bnb-4bit"`로 바꿔 다시 실행.

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # 느리면 unsloth/Qwen2.5-3B-Instruct-bnb-4bit 로 교체

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


## 5) 데이터셋 구성
`train.jsonl`의 각 라인은 `{"messages": [user, assistant]}` 형태 — Qwen chat template에 맞춰 토큰화한다.

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": "train.jsonl"})["train"]

def formatting_func(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = raw.map(formatting_func, remove_columns=raw.column_names)
print(train_dataset[0]["text"][:500])


## 6) 학습 (SFTTrainer)
예상 소요: T4에서 7B 모델 기준 508건 x 3 epoch → 대략 15~30분.

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="outputs",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=sft_config,
)

trainer_stats = trainer.train()


## 7) 어댑터 저장
로컬 + (마운트했다면) Drive 둘 다에 저장.

In [ ]:
LOCAL_DIR = "qlora_adapter"
model.save_pretrained(LOCAL_DIR)
tokenizer.save_pretrained(LOCAL_DIR)

import shutil, os
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copytree(LOCAL_DIR, SAVE_DIR, dirs_exist_ok=True)
    print(f"Drive에 저장됨: {SAVE_DIR}")
else:
    print("Drive 미연결 — 로컬(qlora_adapter/)에만 저장됨, 세션 종료 시 사라짐에 주의")


## 8) 평가 헬퍼 함수
`experiments/run_l2_accuracy.py`의 CLASSIFY_PROMPT/NOVEL_ERRORS와 동일한 기준 — Groq 기록(Category 92%/Action 96%, n=50)과 직접 비교 가능.

In [ ]:
import json, re, time

FastLanguageModel.for_inference(model)

def predict(log_text: str) -> dict:
    prompt = INSTRUCTION.format(log=log_text[:600])
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    out = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.0, do_sample=False, use_cache=True)
    latency_ms = (time.perf_counter() - t0) * 1000
    gen = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    try:
        start, end = gen.find("{"), gen.rfind("}") + 1
        obj = json.loads(gen[start:end]) if start >= 0 and end > start else {}
    except Exception:
        obj = {}
    return {
        "category": obj.get("category", "PARSE_ERROR"),
        "action": obj.get("action", "PARSE_ERROR"),
        "target_process": obj.get("target_process"),
        "latency_ms": latency_ms,
    }


In [ ]:
INSTRUCTION = 'You are a log classification expert for MLOps systems.\nGiven an error log, output EXACTLY this JSON format and nothing else:\n{{"category": "<CATEGORY>", "action": "<ACTION>", "target_process": "<PROCESS or null>"}}\n\nCategories: Out_Of_Memory, Memory_Leak, CPU_Overload, DB_Connection, DB_Timeout, DB_Deadlock, Network_Timeout, Network_Unreachable, Permission_Denied, Configuration_Error, Auth_Error, Path_Not_Found, Disk_Full, Process_Crash, Port_Conflict, Unknown\n\nActions: clear_memory, restart_service, kill_process, escalate_to_human, execute_llm_command, execute_rule_command, alert_only\n\nError log:\n{log}\n\nJSON:'

ACTION_MAP = {
    "Out_Of_Memory": "clear_memory",
    "Memory_Leak": "restart_service",
    "Network_Timeout": "restart_service",
    "DB_Connection": "restart_service",
    "Configuration_Error": "escalate_to_human",
    "Permission_Denied": "escalate_to_human",
    "Disk_Full": "clear_memory",
    "Process_Crash": "restart_service",
    "Port_Conflict": "restart_service",
    "Auth_Error": "escalate_to_human"
}

NOVEL_ERRORS = [
    {
        "log": "FATAL: Torch CUDA out of memory. Tried to allocate 3.50 GiB (GPU 0; 10.76 GiB total capacity; 8.92 GiB already allocated). Consider reducing batch_size.",
        "category": "Out_Of_Memory"
    },
    {
        "log": "java.lang.OutOfMemoryError: GC overhead limit exceeded at org.apache.spark.executor.Executor$TaskRunner.run(Executor.scala:412)",
        "category": "Out_Of_Memory"
    },
    {
        "log": "MemoryError: Unable to allocate 14.2 GiB for array with shape (1920000000,) and data type float64",
        "category": "Out_Of_Memory"
    },
    {
        "log": "ERROR kernel: Out of memory: Kill process 18743 (gunicorn) score 892 or sacrifice child. Killed process 18743 total-vm:9845328kB, anon-rss:7612400kB",
        "category": "Out_Of_Memory"
    },
    {
        "log": "RuntimeError: CUDA out of memory. Tried to allocate 512 MiB. GPU 0 has a total capacity of 8.00 GiB of which 312 MiB is free.",
        "category": "Out_Of_Memory"
    },
    {
        "log": "WARN  MemoryManager: RSS memory 14.8GB exceeds soft limit 12GB. Heap growth detected over last 2h: +340MB/min. Possible memory leak in DataLoader workers.",
        "category": "Memory_Leak"
    },
    {
        "log": "WARNING: python3 process memory usage: 11.2 GB (was 2.1 GB 3 hours ago). GC collections: gen0=142891 gen1=312 gen2=0. Leak suspected in cache layer.",
        "category": "Memory_Leak"
    },
    {
        "log": "ALERT: celery worker PID 7821 resident set size grew from 512MB to 9.8GB over 6 hours without GC release. Restarting worker recommended.",
        "category": "Memory_Leak"
    },
    {
        "log": "HeapDump triggered: live objects count increased by 2.3M in last 30min. Dominant type: byte[] (87%). Possible off-heap memory leak in netty pipeline.",
        "category": "Memory_Leak"
    },
    {
        "log": "mlflow tracking server: memory increased 50MB/request, no release observed. Top allocator: artifact cache (LRUCache unbounded). OOM expected within 2h.",
        "category": "Memory_Leak"
    },
    {
        "log": "requests.exceptions.ConnectTimeout: HTTPSConnectionPool(host='s3.us-east-1.amazonaws.com', port=443): Max retries exceeded. Connect timeout=30s",
        "category": "Network_Timeout"
    },
    {
        "log": "ERROR grpc: connection to inference-server:50051 timeout after 15000ms. Retries=3 exhausted. Last error: DeadlineExceeded status=4",
        "category": "Network_Timeout"
    },
    {
        "log": "TimeoutError: Ray remote call to worker@192.168.10.45 did not complete within 120s. Task: preprocess_batch. Worker appears unresponsive.",
        "category": "Network_Timeout"
    },
    {
        "log": "socket.timeout: timed out waiting for Kafka broker response (bootstrap.servers=kafka:9092, timeout.ms=30000). Topic: ml-predictions offset lag: 84293",
        "category": "Network_Timeout"
    },
    {
        "log": "urllib3.exceptions.ReadTimeoutError: HTTPConnectionPool(host='feature-store', port=8080): Read timed out. (read timeout=45) after sending GET /features/batch",
        "category": "Network_Timeout"
    },
    {
        "log": "sqlalchemy.exc.OperationalError: (psycopg2.OperationalError) SSL connection has been closed unexpectedly. host=pgbouncer port=6432 dbname=mlops_prod",
        "category": "DB_Connection"
    },
    {
        "log": "ERROR [connection-pool] All 20 connections in pool exhausted. Waiting threads: 47. PostgreSQL max_connections=100 (85 used by other services).",
        "category": "DB_Connection"
    },
    {
        "log": "pymongo.errors.ServerSelectionTimeoutError: mongodb://mongo-primary:27017: [Errno 111] Connection refused, Timeout: 30s, Topology: ReplicaSetNoPrimary",
        "category": "DB_Connection"
    },
    {
        "log": "redis.exceptions.ConnectionError: Error 104 connecting to redis-sentinel:26379. Connection reset by peer. Sentinel failover in progress.",
        "category": "DB_Connection"
    },
    {
        "log": "ERROR Cassandra: All host(s) tried for query failed. Last host tried: 10.0.1.15:9042 ([Errno 110] Connection timed out). Keyspace: ml_features",
        "category": "DB_Connection"
    },
    {
        "log": "yaml.scanner.ScannerError: mapping values are not allowed here in 'config/training.yaml', line 47, column 18. Check indentation near 'learning_rate:'",
        "category": "Configuration_Error"
    },
    {
        "log": "ConfigurationError: Invalid value for BATCH_SIZE: 'auto' (expected int). Check environment variable or config/defaults.yaml batch_size field.",
        "category": "Configuration_Error"
    },
    {
        "log": "ERROR: Failed to parse Hydra config. OmegaConf error at model.architecture.layers[2]: value '256x' cannot be converted to int",
        "category": "Configuration_Error"
    },
    {
        "log": "toml.decoder.TomlDecodeError: Found invalid character in key name: ' '. (line 23 column 5 char 412) in pyproject.toml [tool.mlflow] section",
        "category": "Configuration_Error"
    },
    {
        "log": "jsonschema.exceptions.ValidationError: 'dropout_rate' is a required property in schema 'ModelConfig'. Check model_config.json missing field.",
        "category": "Configuration_Error"
    },
    {
        "log": "PermissionError: [Errno 13] Permission denied: '/mnt/nfs/checkpoints/model_v3/epoch_45.ckpt'. Current user: mlops-svc (uid=1001). Required: write on /mnt/nfs",
        "category": "Permission_Denied"
    },
    {
        "log": "ERROR s3: AccessDenied: User: arn:aws:iam::123456789:role/ml-training is not authorized to perform: s3:PutObject on resource: arn:aws:s3:::prod-models/*",
        "category": "Permission_Denied"
    },
    {
        "log": "subprocess.CalledProcessError: EACCES: permission denied, open '/var/log/mlflow/tracking.log'. Process uid=1002 requires group 'mlflow-log' membership.",
        "category": "Permission_Denied"
    },
    {
        "log": "kubectl: Error from server (Forbidden): pods is forbidden: User 'system:serviceaccount:ml-team:trainer' cannot create resource 'pods' in namespace 'gpu-pool'",
        "category": "Permission_Denied"
    },
    {
        "log": "ERROR docker: Got permission denied while trying to connect to Docker daemon socket /var/run/docker.sock. Add user 'mlops' to 'docker' group.",
        "category": "Permission_Denied"
    },
    {
        "log": "OSError: [Errno 28] No space left on device: '/data/mlflow/artifacts/run_a3f9b2/checkpoints/epoch_200.pt'. Disk usage: /data 99.8% (1.8T/1.8T)",
        "category": "Disk_Full"
    },
    {
        "log": "ERROR: Cannot write tensorboard event file. [Errno 28] No space left on device at /tmp/tensorboard_logs. Free: 0 bytes on /tmp (tmpfs 8G full).",
        "category": "Disk_Full"
    },
    {
        "log": "FATAL: PostgreSQL could not write to file 'pg_wal/000000010000003A': No space left on device. WAL segment creation failed. Database halted.",
        "category": "Disk_Full"
    },
    {
        "log": "docker: Error response from daemon: failed to create layer: apply layer: ApplyLayer: write /var/lib/docker/overlay2: no space left on device.",
        "category": "Disk_Full"
    },
    {
        "log": "DiskFullError: Spark shuffle write failed on executor 12. Path: /local/disk1/spark-shuffle/. Filesystem 97% full (485G/500G). Executor will be removed.",
        "category": "Disk_Full"
    },
    {
        "log": "ERROR: Training worker PID 23841 terminated with signal 11 (SIGSEGV). Core dumped to /tmp/core.23841. Last op: torch::autograd::AccumulateGrad",
        "category": "Process_Crash"
    },
    {
        "log": "CRITICAL: Celery worker process exited with code 137 (SIGKILL/OOM). Task ml_inference.predict_batch was lost. Re-queuing with retry=2.",
        "category": "Process_Crash"
    },
    {
        "log": "fatal error: unexpected signal during runtime execution. SIGBUS at PC=0x7f3a9c000000 sp=0x7ffe12340000. Go runtime crashed in model serving goroutine.",
        "category": "Process_Crash"
    },
    {
        "log": "ERROR supervisor: gunicorn worker with pid 9923 died. Stacktrace: Segmentation fault (core dumped). Respawning. Crashes in last 1h: 5",
        "category": "Process_Crash"
    },
    {
        "log": "WATCHDOG: Process 'model-server' (PID 4412) killed — health check failed 3 consecutive times. Exit code: -9. Restarting with backoff 30s.",
        "category": "Process_Crash"
    },
    {
        "log": "OSError: [Errno 98] Address already in use. Failed to bind TensorBoard to 0.0.0.0:6006. Port occupied by PID 8823 (python3 train.py --tensorboard)",
        "category": "Port_Conflict"
    },
    {
        "log": "ERROR: MLflow UI failed to start: [Errno 98] EADDRINUSE: address already in use :::5000. Try: mlflow ui --port 5001",
        "category": "Port_Conflict"
    },
    {
        "log": "uvicorn.error: [Errno 98] error while attempting to bind on address ('0.0.0.0', 8080): address already in use. Previous instance still running?",
        "category": "Port_Conflict"
    },
    {
        "log": "FATAL: Ray head node cannot start GCS on port 6379: bind: address already in use. Redis may already be occupying port 6379.",
        "category": "Port_Conflict"
    },
    {
        "log": "Error: Jupyter Lab port 8888 is already in use. Kill the process using: lsof -ti:8888 | xargs kill -9, then restart.",
        "category": "Port_Conflict"
    },
    {
        "log": "AuthenticationError: Weights & Biases API key invalid or expired. Token: 'wand_xxxxxx...'. Re-authenticate with: wandb login --relogin",
        "category": "Auth_Error"
    },
    {
        "log": "ERROR mlflow: 401 Unauthorized: Bearer token expired for tracking server https://mlflow.internal. Re-run: mlflow.set_tracking_uri() with fresh token.",
        "category": "Auth_Error"
    },
    {
        "log": "google.auth.exceptions.TransportError: 403 Forbidden. Service account ml-trainer@project.iam.gserviceaccount.com lacks role roles/storage.objectAdmin on bucket gs://ml-data",
        "category": "Auth_Error"
    },
    {
        "log": "JWTError: Signature verification failed. Token issued at 2026-05-10T09:00:00Z expired at 2026-05-10T10:00:00Z. Current time: 2026-05-22T14:30:00Z. Re-login required.",
        "category": "Auth_Error"
    },
    {
        "log": "ERROR: Vault token renewal failed: permission denied. Policy 'ml-secrets-read' does not allow token renewal. Contact admin to re-issue token.",
        "category": "Auth_Error"
    }
]


## 9) 벤치마크 1 — NOVEL_ERRORS (n=50, Groq와 동일 기준으로 직접 비교)

In [ ]:
cat_correct = act_correct = 0
latencies = []
per_cat = {}

for item in NOVEL_ERRORS:
    true_cat = item["category"]
    true_act = ACTION_MAP[true_cat]
    pred = predict(item["log"])
    latencies.append(pred["latency_ms"])
    cat_ok = pred["category"] == true_cat
    act_ok = pred["action"] == true_act
    cat_correct += cat_ok
    act_correct += act_ok
    per_cat.setdefault(true_cat, []).append(cat_ok)

n = len(NOVEL_ERRORS)
print(f"Category Accuracy: {cat_correct/n*100:.1f}% ({cat_correct}/{n})")
print(f"Action Accuracy  : {act_correct/n*100:.1f}% ({act_correct}/{n})")
print(f"평균 지연    : {sum(latencies)/len(latencies):.0f}ms")
for cat, vals in per_cat.items():
    print(f"  {cat:<22} {sum(vals)/len(vals)*100:>5.1f}%")

novel_summary = {
    "n_samples": n,
    "cat_accuracy": round(cat_correct/n, 4),
    "act_accuracy": round(act_correct/n, 4),
    "avg_latency_ms": round(sum(latencies)/len(latencies), 1),
}


## 10) 벤치마크 2 — 홀드아웃 eval.jsonl (n=614, in-distribution, 더 큰 표본)

In [ ]:
eval_records = [json.loads(l) for l in open("eval.jsonl", encoding="utf-8")]

def parse_true(rec):
    assistant_json = json.loads(rec["messages"][1]["content"])
    return assistant_json

eval_cat_correct = eval_act_correct = 0
for rec in eval_records:
    log_line = rec["messages"][0]["content"].split("Error log:\n")[1].split("\n\nJSON:")[0]
    true = parse_true(rec)
    pred = predict(log_line)
    eval_cat_correct += pred["category"] == true["category"]
    eval_act_correct += pred["action"] == true["action"]

n_eval = len(eval_records)
print(f"[Held-out eval.jsonl] Category Accuracy: {eval_cat_correct/n_eval*100:.1f}% ({eval_cat_correct}/{n_eval})")
print(f"[Held-out eval.jsonl] Action Accuracy  : {eval_act_correct/n_eval*100:.1f}% ({eval_act_correct}/{n_eval})")

heldout_summary = {
    "n_samples": n_eval,
    "cat_accuracy": round(eval_cat_correct/n_eval, 4),
    "act_accuracy": round(eval_act_correct/n_eval, 4),
}


## 11) 최종 요약 저장 + 다운로드
이 JSON을 다운로드해서 Claude에게 결과를 전달하면 된다.

In [ ]:
summary = {
    "model": model_name,
    "novel_errors_benchmark": novel_summary,
    "heldout_eval": heldout_summary,
    "groq_baseline_reference": {
        "cat_accuracy": 0.92, "act_accuracy": 0.96, "n_samples": 50,
        "source": "experiments/results/l2_accuracy_summary_groq.json (2026-08-27)",
    },
}
with open("qlora_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))

from google.colab import files
files.download("qlora_eval_summary.json")
